In [2]:
!pip install PyWavelets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 7.3 MB/s  0:00:00 eta 0:00:01


In [5]:
import numpy as np
import pywt

# Load segmented beats
X = np.load("X_beats.npy")
y = np.load("y_labels.npy")

# Configuration
WAVELET = "db4"
LEVEL = 4

# Feature extractors
def energy(coeff):
    return np.sum(coeff**2)

def entropy(coeff):
    p = np.abs(coeff) / np.sum(np.abs(coeff))
    return -np.sum(p * np.log2(p + 1e-12))

def rms(coeff):
    return np.sqrt(np.mean(coeff**2))


def extract_features_from_coeff(coeff):
    """Extract 6 features from a wavelet coefficient array."""
    return np.array([
        energy(coeff),
        entropy(coeff),
        np.mean(coeff),
        np.var(coeff),
        np.std(coeff),
        rms(coeff)
    ])


# MAIN FEATURE EXTRACTION FUNCTION
def extract_wavelet_features(beat):
    # wavelet decomposition: A4, D4, D3, D2, D1
    coeffs = pywt.wavedec(beat, WAVELET, level=LEVEL)

    # coeffs[0] = A4
    # coeffs[1] = D4
    # coeffs[2] = D3
    # coeffs[3] = D2
    # coeffs[4] = D1

    features = []

    for c in coeffs:
        feat = extract_features_from_coeff(c)
        features.append(feat)

    # Flatten all features into 1D vector
    return np.concatenate(features)



# Extract features for ALL beats
feature_list = []

print("Extracting DWT features for", len(X), "beats...")

for beat in X:
    f = extract_wavelet_features(beat)
    feature_list.append(f)

X_wavelet = np.array(feature_list)

print("DWT feature matrix shape:", X_wavelet.shape)

# Save final feature dataset
np.save("X_wavelet_features.npy", X_wavelet)
np.save("y_labels.npy", y)

print("Saved: X_wavelet_features.npy")


Extracting DWT features for 101418 beats...
DWT feature matrix shape: (101418, 30)
Saved: X_wavelet_features.npy
